In [ ]:
# two pretrained models as a check on notebook 04: Chronos-2 on the 'when', TabICLv2 on the 'who'. Kernel: Python (yjmob-gpu)
import os, sys, time, json
from pathlib import Path
ROOT = Path("/projects/ancz7294/yjmob")
os.environ.setdefault("HF_HOME", str(ROOT.parent / ".cache/huggingface"))       # model weights in /projects, not the small home folder
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score, confusion_matrix
PROC, FIG = ROOT / "data/processed", ROOT / "figures"
SEED = 20260917
C = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
plt.rcParams.update({"figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "axes.edgecolor": "#c3c2b7",
    "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.6, "axes.spines.top": False, "axes.spines.right": False,
    "text.color": "#0b0b0b", "axes.labelcolor": "#52514e", "xtick.color": "#898781", "ytick.color": "#898781",
    "font.family": "sans-serif", "font.size": 10, "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "figure.dpi": 110, "savefig.dpi": 150, "savefig.bbox": "tight"})
def save(fig, name): fig.savefig(FIG / f"{name}.png"); plt.show(); plt.close(fig)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")
days = pd.read_parquet(PROC / "day_table.parquet").set_index("d").sort_index()
lab = pd.read_parquet(PROC / "person_breaks.parquet")
shares = pd.read_parquet(PROC / "person_day_shares.parquet")
ORDER = ["far roamers", "mid-range movers", "regular commuters", "local regulars", "weekend movers", "homebodies"]
print(f"{shares.uid.nunique():,} people x 75 days | label for {len(lab):,} people, {lab.broke.mean():.1%} broke")

In [ ]:
# Chronos-2 (Amazon, Apache-2.0), zero-shot: from each person's own history it forecasts two daily numbers, the share of seen half
# hours at home and at the daytime place. Two forecasts of 15 days each, made from days 0-44 and from days 0-59, so the ordinary and the
# emergency fortnight are forecast from the same distance. A day the forecast misses badly is a day the routine did not follow its past.
from chronos import Chronos2Pipeline
s = shares.copy()
s["home"] = np.where(s.seen >= 4, s.home / s.seen.clip(lower=1), np.nan); s["place"] = np.where(s.seen >= 4, s.place / s.seen.clip(lower=1), np.nan)
ts = pd.date_range("2000-01-01", periods=75, freq="D")                          # only an index: the data has no calendar
s["timestamp"] = ts[s.d.to_numpy()]
s["workday"] = days.workday.reindex(s.d).to_numpy().astype(float); s["weekend"] = days.weekend.reindex(s.d).to_numpy().astype(float)
s = s[s.uid.isin(lab.index)].rename(columns={"uid": "id"})[["id", "timestamp", "home", "place", "workday", "weekend"]]
pipe = Chronos2Pipeline.from_pretrained("amazon/chronos-2", device_map=DEVICE)
t0, out = time.time(), []
for origin in [44, 59]:
    ctx = s[s.timestamp <= ts[origin]]
    fut = s[(s.timestamp > ts[origin]) & (s.timestamp <= ts[origin + 15])][["id", "timestamp", "workday", "weekend"]]
    out.append(pipe.predict_df(ctx, future_df=fut, id_column="id", timestamp_column="timestamp", target=["home", "place"],
                               prediction_length=15, quantile_levels=[0.1, 0.5, 0.9], batch_size=256))
pr = pd.concat(out)
print(f"Chronos-2: {pr.id.nunique():,} people, {len(pr):,} forecasts in {time.time() - t0:.0f} s on {DEVICE}")
act = s.set_index(["id", "timestamp"])[["home", "place"]].stack().rename("y").reset_index().rename(columns={"level_2": "target_name"})
m = pr.merge(act, on=["id", "timestamp", "target_name"], how="inner")
m["loss"] = sum(np.maximum(q * (m.y - m[str(q)]), (q - 1) * (m.y - m[str(q)])) for q in [0.1, 0.5, 0.9]) / 3   # quantile loss
m["d"] = (m.timestamp - ts[0]).dt.days
dl = m.groupby(["id", "d"]).loss.mean().reset_index(); dl["y"] = (dl.d >= 60).astype(int)
dl["split"] = lab.split.reindex(dl.id).to_numpy(); dl["kind"] = lab.kind.reindex(dl.id).to_numpy()
ev = dl[dl.split == "eval"]
print(f"day level, evaluation people: AUC of the forecast loss for emergency vs ordinary days {roc_auc_score(ev.y, ev.loss):.3f}")
per = dl.groupby(["id", "y"]).loss.mean().unstack().dropna(); per["chronos_score"] = per[1] - per[0]
per = per.join(lab[["broke", "delta", "kind", "split"]])
pe_ = per[per.split == "eval"]
print(f"emergency fortnight forecast worse for {np.mean(pe_.chronos_score > 0):.3f} of evaluation people")
print(f"agreement with the habit-model label (evaluation people): correlation with the rise in surprise {np.corrcoef(pe_.chronos_score, pe_.delta)[0, 1]:.3f},"
      f" AUC of the Chronos score for 'broke' {roc_auc_score(pe_.broke, pe_.chronos_score):.3f}")
print(pe_.groupby("kind").chronos_score.median().reindex(ORDER).round(4).to_string())
nb = pd.read_csv(PROC / "nb_detectors_eval.csv"); nbm = pd.read_csv(PROC / "nb_models_eval.csv", index_col=0)
auc = pd.Series({"multinomial NB": nbm.loc["multinomial (where and when)", "AUC"], "Bernoulli NB (when seen)": nbm.loc["Bernoulli (when seen)", "AUC"],
                 "where NB, everyone": nb[(nb.kind == "all") & (nb.detector == "one for everyone")].AUC.iloc[0],
                 "where NB, per kind": nb[(nb.kind == "all") & (nb.detector == "one per kind")].AUC.iloc[0],
                 "where NB, per person": nb[(nb.kind == "all") & (nb.detector == "one per person")].AUC.iloc[0],
                 "Chronos-2, zero-shot": roc_auc_score(ev.y, ev.loss)})
print(auc.round(3).to_string())
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), gridspec_kw={"width_ratios": [1, 1.3]})
axes[0].barh(auc.index[::-1], auc.values[::-1] * 100, color=[C[1] if "Chronos" in i else C[0] for i in auc.index[::-1]])
axes[0].axvline(50, color="#52514e", ls="--", lw=1); axes[0].set(xlim=(45, max(65, auc.max() * 100 + 2)), xlabel="AUC (%), one day at a time", title="Telling an emergency day from an ordinary one")
axes[0].yaxis.grid(False)
for k in ORDER:
    g = ev[ev.kind == k].groupby("d").loss.mean(); axes[1].plot(g.index, g.values, lw=1.6, label=k)
axes[1].axvspan(59.5, 74.5, color=C[1], alpha=0.07, lw=0)
axes[1].set(xlabel="day", ylabel="Chronos-2 forecast loss", title="How wrong the forecast is, day by day (evaluation people)"); axes[1].legend(frameon=False, fontsize=8, ncol=2)
save(fig, "f45_chronos")

In [ ]:
# TabICLv2 (2026, BSD licence, open weights): a model pretrained on millions of synthetic tables, which reads the training people as
# examples and predicts the test people in one pass, without training on them. Same features, same people as the trees
from tabicl import TabICLClassifier
T = pd.read_parquet(PROC / "who_breaks_table.parquet")
feat = [c for c in T.columns if c not in ("broke", "split", "kind")]
tr, te = (T.split == "dev").to_numpy(), (T.split == "eval").to_numpy()
Xtr, Xte, ytr, yte = T.loc[tr, feat].to_numpy("float32"), T.loc[te, feat].to_numpy("float32"), T.broke[tr].to_numpy(), T.broke[te].to_numpy()
t0 = time.time()
clf = TabICLClassifier(device=DEVICE, random_state=SEED).fit(Xtr, ytr)
p_icl = clf.predict_proba(Xte)[:, 1]
print(f"TabICLv2: {len(Xtr):,} training people as examples, {len(Xte):,} predicted in {time.time() - t0:.0f} s on {DEVICE}")
cut = ytr.mean()                                    # call 'broke' above the training share, the same balance the trees' class weights give
trees = pd.read_parquet(PROC / "tree_test_scores.parquet")
cmp = {nm: (trees[nm].to_numpy(), 0.5) for nm in trees.columns if nm != "y"}
cmp["TabICLv2"] = (p_icl, cut)
rows = [(nm, accuracy_score(yte, p > c), balanced_accuracy_score(yte, p > c), roc_auc_score(yte, p)) for nm, (p, c) in cmp.items()]
res = pd.DataFrame(rows, columns=["model", "accuracy", "balanced accuracy", "AUC"]).set_index("model")
print(res.round(3).to_string()); print(f"always 'held': accuracy {1 - yte.mean():.3f}, balanced accuracy 0.5, AUC 0.5")
fig, axes = plt.subplots(1, 2, figsize=(12.5, 3.9), gridspec_kw={"width_ratios": [1.5, 1]})
xx = np.arange(len(res))
axes[0].bar(xx - 0.2, res["balanced accuracy"] * 100, width=0.4, color=C[0], label="balanced accuracy")
axes[0].bar(xx + 0.2, res["AUC"] * 100, width=0.4, color=C[1], label="AUC")
axes[0].axhline(50, color="#52514e", ls="--", lw=1); axes[0].set_xticks(xx); axes[0].set_xticklabels([n.replace(", ", "\n") for n in res.index], fontsize=9)
axes[0].set(ylim=(40, 75), ylabel="%", title="Who breaks: readable trees against a pretrained model"); axes[0].legend(frameon=False, fontsize=9); axes[0].xaxis.grid(False)
cm = confusion_matrix(yte, p_icl > cut)
axes[1].imshow(cm, cmap="Blues", vmin=0, vmax=cm.max() * 1.3); axes[1].grid(False)
for (r, c_), v in np.ndenumerate(cm): axes[1].text(c_, r, f"{v:,}\n{v / cm[r].sum():.0%} of row", ha="center", va="center", fontsize=9)
axes[1].set_xticks([0, 1]); axes[1].set_xticklabels(["held", "broke"]); axes[1].set_yticks([0, 1]); axes[1].set_yticklabels(["held", "broke"])
axes[1].set(xlabel="predicted", ylabel="actual", title="TabICLv2, test people")
save(fig, "f46_tabicl")

In [ ]:
per.to_parquet(PROC / "chronos_scores.parquet"); pd.DataFrame({"y": yte, "tabicl": p_icl}).to_parquet(PROC / "tabicl_test_scores.parquet")
log = {"device": DEVICE, "chronos_day_auc_eval": round(float(auc["Chronos-2, zero-shot"]), 4),
       "chronos_agrees_with_label_auc": round(float(roc_auc_score(pe_.broke, pe_.chronos_score)), 4), "who_breaks": res.round(4).to_dict()}
(ROOT / "logs/part3_foundation_log.json").write_text(json.dumps(log, indent=2)); print(json.dumps(log, indent=2))

In [ ]:
# one archive with everything notebooks 04 and 05 produced, to download from the file browser (save both notebooks first)
import zipfile
out = ROOT / "part3_outputs.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(FIG.glob("f3[3-9]_*.png")) + sorted(FIG.glob("f4[0-6]_*.png")) + sorted(FIG.glob("t1[1-6]_*.png")):
        z.write(p, f"figures/{p.name}")
    for p in [ROOT / "code/breaks.py", ROOT / "logs/part3_log.json", ROOT / "logs/part3_foundation_log.json",
              ROOT / "data/samples/nb_days_sample.csv", ROOT / "data/samples/who_breaks_sample.csv",
              ROOT / "notebooks/04_when_and_who.ipynb", ROOT / "notebooks/05_foundation_models.ipynb"]:
        if p.exists(): z.write(p, str(p.relative_to(ROOT)))
        else: print("missing:", p)
print(out, f"{out.stat().st_size / 1e6:.1f} MB")